# Trabajo Práctico Aprendizaje Automático 2 - Predicción de puntuación y capitalización en texto normalizado

In [1]:
import re
from datasets import load_dataset
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from transformers import BertModel, BertTokenizer
from sklearn.model_selection import train_test_split   
import unicodedata


/home/mariok99/miniconda3/envs/metricas-nlp/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Seteamos la semilla para que los resultados sean reproducibles
SEED = 2026
np.random.seed(SEED)

In [3]:
model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2115.20it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p

## Datasets

Parlamento europeo dataset (parte en español): https://statmt.org/europarl/

In [4]:
def tiene_pregunta(ejemplo):
    texto = ejemplo["translation"]["es"]
    return "¿" in texto

In [5]:
dataset = load_dataset(
    "Helsinki-NLP/europarl",
    "en-es",
    split="train",
    streaming=True,
)

N_PREG = 2000
N_NO_PREG = 10000

preguntas = []
no_preguntas = []

for ejemplo in dataset:
    print(ejemplo)
    if len(ejemplo["translation"]["es"].split(" ")) < 12: continue
    if tiene_pregunta(ejemplo):
        if len(preguntas) < N_PREG:
            preguntas.append(ejemplo["translation"]["es"])
    else:
        if len(no_preguntas) < N_NO_PREG:
            no_preguntas.append(ejemplo["translation"]["es"])
    # Cortar cuando ya tengamos ambos buckets completos
    if len(preguntas) >= N_PREG and len(no_preguntas) >= N_NO_PREG:
        break


KeyboardInterrupt: 

In [ ]:
df_preguntas = pd.DataFrame({"Text": preguntas})
df_no_preguntas = pd.DataFrame({"Text": no_preguntas})

df_balanceado = pd.concat(
    [df_preguntas, df_no_preguntas], ignore_index=True
).sample(frac=1, random_state=2026).reset_index(drop=True)

print(f" Tamaño DataFrame Preguntas: {len(df_preguntas)}\n", f"Tamaño DataFrame NO Preguntas: {len(df_no_preguntas)}\n", f"Tamaño DataFrame Balanceado: {len(df_balanceado)}\n")

In [ ]:
df_balanceado.head(1)

# Construcción dataset

In [11]:
VOCALES = set("aeiouáéíóúü")
VOCALES_SIN_TILDE = set("aeiouü")

ADVERBIOS_INTERROGATIVOS = {
    "qué", "quién", "quiénes", "cómo", "cuándo", "dónde",
    "cuánto", "cuánta", "cuántos", "cuántas", "cuál", "cuáles",
}

PRONOMBRES_INTERROGATIVOS = {
    "qué", "quién", "quiénes", "cuál", "cuáles", "cuánto",
    "cuánta", "cuántos", "cuántas",
}

# Unimos ambos para el atributo "es_palabra_interrogativa"
PALABRAS_INTERROGATIVAS = ADVERBIOS_INTERROGATIVOS | PRONOMBRES_INTERROGATIVOS



def _es_palabra_interrogativa(palabra: str) -> int:
    return int(palabra in PALABRAS_INTERROGATIVAS)

def fijarsesitienecoso(oracion):
    superoraciones = oracion.split(" ")
    for palabra in superoraciones:
        
        if palabra in ADVERBIOS_INTERROGATIVOS:
            return True
    return False

In [12]:
# Función de Etiquetado
def construir_dataset_etiquetado(oraciones):
    filas = []
    
    for id_oracion, oracion in enumerate(oraciones):
        palabras_crudas = oracion.split(" ")
        
        for id_palabra, palabra_cruda in enumerate(palabras_crudas):

            palabra_sin_signos = re.sub(r'[¿?.,]', '', palabra_cruda)
            
            if not palabra_sin_signos:
                continue

            empieza_con_mayuscula = int(palabra_sin_signos[0].isupper())
            palabra_limpia = palabra_sin_signos.lower()
            
            tienecoso = fijarsesitienecoso(oracion)
            
            filas.append({
                'id_oracion': id_oracion,
                'id_palabra': id_palabra,
            'tienecosoenlaoracion':tienecoso,
                'palabra' : palabra_cruda,'palabra_limpia': palabra_limpia,
                'tiene_¿': int('¿' in palabra_cruda),
                'tiene_?': int('?' in palabra_cruda),
                'tiene_.': int('.' in palabra_cruda),
                'tiene_,': int(',' in palabra_cruda),
                'empieza_con_mayuscula': empieza_con_mayuscula
            })
            
    return pd.DataFrame(filas)
                

In [ ]:
corpus_prueba = [
    "Dale Tomi, llegamos tarde. cuándo",
    "Apurate.",
    "Día"
]

construir_dataset_etiquetado(corpus_prueba)

In [14]:
# Función de Atributos
# advervio de pregunta
# Atributos propuestos:
#    - distancia al inicio de la oración (es trampa?)
#    - tiene_tilde
#    - cantidad_vocales
#    - cantidad_consonantes
#    - es_adverbio_interrogativo
#    - lo mismo con la anterior y la siguiente 
# 
# df tiene la estructura de construir_dataset_etiquetado

def _tiene_tilde(palabra: str) -> int:
    """1 si la palabra contiene al menos una vocal con tilde (á é í ó ú)."""
    return int(any(ch in "áéíóú" for ch in palabra))

def _cantidad_vocales(palabra: str) -> int:
    return sum(1 for ch in palabra if ch in VOCALES)


def _cantidad_consonantes(palabra: str) -> int:
    """Cuenta letras que no son vocales. Ignora dígitos, espacios, etc."""
    return sum(
        1 for ch in palabra
        if unicodedata.category(ch).startswith("L") and ch not in VOCALES
    )



# ---------------------------------------------------------------------------
# Función principal
# ---------------------------------------------------------------------------

def añadir_atributos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega atributos léxicos y contextuales a un DataFrame producido por
    `construir_dataset_etiquetado`.

    Columnas de entrada esperadas:
        id_oracion, id_palabra, palabra_limpia,
        tiene_¿, tiene_?, tiene_., tiene_,, empieza_con_mayuscula

    Columnas agregadas:
        distancia_inicio          : posición de la palabra dentro de la oración
        tiene_tilde               : 1 si tiene vocal acentuada
        cantidad_vocales          : vocales en la palabra
        cantidad_consonantes      : consonantes en la palabra
        cantidad_letras           : vocales + consonantes
        es_adverbio_interrogativo : 1 si es qué/quién/cómo/cuándo/dónde/...
        es_palabra_interrogativa  : idem (alias más amplio; incluye pronombres)
        prev_es_interrogativa     : 1 si la palabra anterior es interrogativa
        next_es_interrogativa     : 1 si la palabra siguiente es interrogativa
        prev_tiene_tilde          : 1 si la palabra anterior tiene tilde
        next_tiene_tilde          : 1 si la palabra siguiente tiene tilde
        prev_tiene_¿              : 1 si la palabra anterior traía '¿'
        next_tiene_?              : 1 si la palabra siguiente trae '?'
    """
    df = df.copy()

    # --- Atributos por palabra (independientes del contexto) ---------------
    df["distancia_inicio"] = df["id_palabra"]

    df["tiene_tilde"] = df["palabra_limpia"].map(_tiene_tilde)
    df["cantidad_vocales"] = df["palabra_limpia"].map(_cantidad_vocales)
    df["cantidad_consonantes"] = df["palabra_limpia"].map(_cantidad_consonantes)
    df["cantidad_letras"] = df["cantidad_vocales"] + df["cantidad_consonantes"]

    df["es_adverbio_interrogativo"] = (
        df["palabra_limpia"].map(_es_palabra_interrogativa)
    )
    df["es_palabra_interrogativa"] = df["es_adverbio_interrogativo"]

    # --- Atributos contextuales (palabra anterior / siguiente) -------------
    # Agrupamos por oración y hacemos shift dentro de cada grupo.
    # fillna(0) cubre los extremos de la oración (primera y última palabra).
    for col in [
        "es_palabra_interrogativa",
        "tiene_tilde",
        "tiene_¿",
        "tiene_?",
    ]:
        df[f"prev_{col}"] = (
            df.groupby("id_oracion")[col].shift(1).fillna(0).astype(int)
        )
        df[f"next_{col}"] = (
            df.groupby("id_oracion")[col].shift(-1).fillna(0).astype(int)
        )

    return df   

# Old

In [ ]:
X_train, y_train, X_control, y_control = train_test_split(df_feats2, 'label')
df_control = pd.concat([X_control, y_control], axis=1)
df_dev = pd.concat([X_train, y_train], axis=1)

In [16]:
def es_signo_puntuacion(token: str) -> bool:
    # True si el token es (o contiene solo) signos de puntuación
    if not token or token.startswith("##"):
        return False
    return all(unicodedata.category(ch).startswith("P") for ch in token)


def construir_dataset_interrogacion(
    oraciones,
    tokenizer,
    *,
    quitar_toda_puntuacion: bool = True,
    max_tokens: int = 512,
    ventana: int | None = None,
):
    """
    Construye una lista de instancias token-a-token para predecir '¿'
    a partir de texto normalizado.

    Cada instancia es un dict:
        {
            'oracion_id': int,
            'token': str,
            'token_id': int,
            'distancia_inicio': int,   # cuántos tokens no-'¿' hay antes
            'label': int,              # 1 si estaba precedido por '¿'
            'ventana': list[int|None], # solo si ventana no es None
        }
    """
    registros = []

    for oracion_id, oracion in enumerate(oraciones):
        # 1) Tokenizar la oración original (con su puntuación)
        tokens = tokenizer.tokenize(oracion)
        if not tokens:
            continue

        # 2) Truncar al límite de BERT
        tokens = tokens[:max_tokens]

        # 3) Construir input limpio + etiquetas
        tokens_limpios = []
        target = []
        for i, tok in enumerate(tokens):
            if tok == "¿":
                continue  # '¿' es el target: nunca va al input
            # opcional: quitamos también ? . , etc. Si
            # si encadenamos los clasificadores creo que podría servir ¿?
            if quitar_toda_puntuacion and es_signo_puntuacion(tok):
                continue  
            tokens_limpios.append(tok)
            # etiqueta: ¿el token anterior (en el original) era '¿'?
            target.append(1 if (i > 0 and tokens[i - 1] == "¿") else 0)

        # Por si no habían tokens ¿?
        if not tokens_limpios:
            continue

        # 4) Convertir a ids (BERT ya tiene embeddings: solo pasamos input_ids)
        ids = tokenizer.convert_tokens_to_ids(tokens_limpios)

        # 5) Armar registros (con o sin ventana de contexto)
        n = len(ids)
        for i in range(n):
            registro = {
                "oracion_id": oracion_id, # esto no lo había pensado pero se ve bien...
                "token": tokens_limpios[i],
                "token_id": ids[i],
                "distancia_inicio": i,
                "label": target[i],
            }
            if ventana is not None:
                vecinos = [
                    ids[j] if 0 <= j < n else None
                    for j in range(i - ventana, i + ventana + 1)
                ]
                registro["ventana"] = vecinos
            registros.append(registro)

    return registros

In [17]:
def es_signo_puntuacion(palabra: str) -> bool:
    if not palabra or palabra.startswith("##"):
        return False
    return all(unicodedata.category(ch).startswith("P") for ch in palabra)


def normalizar_texto(texto: str) -> list[str]:
    """Devuelve la lista de palabras normalizadas (minúsculas, sin puntuación)."""
    # 1) Separar por espacios en blanco (cubre espacios, tabs, saltos de línea)
    palabras = texto.split()

    resultado = []
    for palabra in palabras:
        # 2) Quitar signos de puntuación que estén pegados a la palabra
        #    (al principio, al final o en el medio). No quitamos letras,
        #    números, ni marcas diacríticas (á, ñ, ü...).
        limpia = "".join(
            ch for ch in palabra
            if not unicodedata.category(ch).startswith("P")
        )
        # 3) Descartar si quedó vacía (era solo puntuación, ej. '¿', '?', '...')
        if not limpia:
            continue
        # 4) Minúsculas. OJO: .lower() respeta acentos en Python 3.
        resultado.append(limpia.lower())

    return resultado


def normalizar(texto: str) -> str:
    """Versión string del normalizador: '¿Hola? Tonto.' -> 'hola tonto'."""
    return " ".join(normalizar_texto(texto))

#def construir_dataset_interrogacion(
#    oraciones: list[str],
#    tokenizer,
#    *,
#    quitar_toda_puntuacion: bool = True,
#    max_tokens: int = 512,
#    ventana: int | None = None,
#):
#    """
#    Construye una lista de instancias token-a-token para predecir '¿'
#    a partir de texto normalizado.
#
#    Cada instancia es un dict:
#        {
#            'oracion_id': int,
#            'token': str,
#            'token_id': int,
#            'distancia_inicio': int,   # cuántos tokens no-'¿' hay antes
#            'label': int,              # 1 si estaba precedido por '¿'
#            'ventana': list[int|None], # solo si ventana no es None
#        }
#    """
#    registros = []
#
#    for oracion_id, oracion in enumerate(oraciones):
#        # 1) Tokenizar la oración original (con su puntuación)
#        # 3) Construir input limpio + etiquetas
#        palabras_limpias = []
#        target = []
#        palabras = oracion.split(" ")
#        for i, p in enumerate(palabras):
#            if p in "¿":
#                continue  # '¿' es el target: nunca va al input
#            # opcional: quitamos también ? . , etc. Si
#            # si encadenamos los clasificadores creo que podría servir ¿?
#            if quitar_toda_puntuacion and es_signo_puntuacion(p):
#                continue  
#            palabras_limpias.append(p[1:])
#            # etiqueta: ¿el token anterior (en el original) era '¿'?
#            target.append(1 if (i > 0 and tokens[i - 1] == "¿") else 0)
#
#
#        n = len(palabras)
#        for i in range(n):
#            registro = {
#                "oracion_id": oracion_id, # esto no lo había pensado pero se ve bien...
#                "token": palabras_limpias[i],
#                "token_id": ids[i],
#                "distancia_inicio": i,
#                "label": target[i],
#            }
#            if ventana is not None:
#                vecinos = [
#                    ids[j] if 0 <= j < n else None
#                    for j in range(i - ventana, i + ventana + 1)
#                ]
#                registro["ventana"] = vecinos
#            registros.append(registro)
#
#    return registros

In [ ]:
normalizar("¿Tomi vino gracioso del parcial?, Si.")

In [ ]:
# A) Con ventana de contexto (para modelos no contextuales / baseline)
# se podría usar la lista en sí y no crear df_balanceado

registros_ventana_5 = construir_dataset_interrogacion(
    df_balanceado["Text"].tolist(),
    tokenizer,
    quitar_toda_puntuacion=True,
    ventana=5
)

# C) Convertir a DataFrame para inspeccionar
df_feats = pd.DataFrame(registros_ventana_5)
print(df_feats.head())
print(df_feats["label"].value_counts(normalize=True))  # desbalance esperado

In [ ]:
"¿juancito fue a comer.".split(" ")
x = "abc"
x[0]

In [ ]:
# probar con o sin oracion_id
df_feats2 = df_feats.drop(columns=['token'])

df_feats2

In [ ]:
df_feats2['label']

In [ ]:
print(df_feats2.describe())

In [24]:
from sklearn.model_selection import StratifiedKFold


SEED = 2026
np.random.seed(SEED)

# Definimos el K-Fold estratificado para usar en la validación cruzada de los modelos
stratifiedKFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [25]:
# Función para separar el dataset en train y test, manteniendo la proporción de clases
def train_test_split(df, target):
    buen_pronostico = df[df[target] == 1]
    p = buen_pronostico.shape[0]/df.shape[0]

    mal_pronostico = df[df[target] == 0]
    buenos = len(buen_pronostico)/len(df)
    filasBuenas = int(len(df)*buenos*p)
    filasMalas = int(len(df)*p-filasBuenas)
    # buscar como selecciona .sample()
    test = pd.DataFrame(columns=buen_pronostico.columns).astype(buen_pronostico.dtypes)
    test = pd.concat([test, buen_pronostico.sample(n=filasBuenas, random_state=SEED)], ignore_index=False)
    test = pd.concat([test, mal_pronostico.sample(n=filasMalas, random_state=SEED)], ignore_index=False)

    train = df.drop(test.index, axis=0)    
    return train.drop(columns=[target]), train[target], test.drop(columns=[target]), test[target]

In [26]:
X_train, y_train, X_control, y_control = train_test_split(df_feats2, 'label')
df_control = pd.concat([X_control, y_control], axis=1)
df_dev = pd.concat([X_train, y_train], axis=1)

In [27]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC


def mejor_SVC_dado_df(X_train, y_train):
    param_dist = [
        {
            'estimator__kernel': ['rbf'],
            'estimator__C': np.logspace(-3, 3, 7), 
            'estimator__gamma': list(np.logspace(-4, 1, 6)) + ['scale', 'auto'], 
        },
        {
            'estimator__kernel': ['poly'],
            'estimator__C': np.logspace(-3, 3, 7),
            'estimator__degree': [2, 3, 5],
            'estimator__coef0': [0.0, 0.1, 0.5],
        },
        {
            'estimator__kernel': ['linear'],
            'estimator__C': np.logspace(-3, 3, 7),
        },
    ]

    modelo_base = CalibratedClassifierCV(SVC(), ensemble=False)

    random_search_svc = RandomizedSearchCV(
        modelo_base,
        param_distributions=param_dist,
        n_iter=30,
        scoring='roc_auc',
        cv=stratifiedKFold,
        random_state=SEED,
        n_jobs=-1
    )
    
    random_search_svc.fit(X_train, y_train)
    return random_search_svc

In [ ]:
random_search_svc = mejor_SVC_dado_df(X_train, y_train)
random_search_svc.best_params_

In [ ]:
# Resultados del modelo SVC con su mejor combinación de hiperparámetros más otras métricas de evaluación
resultado_svc = stratifiedKFoldCV(random_search_svc.best_estimator_, X_train, y_train, stratifiedKFold)

resultado_svc.informe

# Consulta
- Sobre añadir datos sinteticos para tener más preguntas ('¿' '?').
- Sobre concatenar oraciones. (Para que el modelo tenga que separarlas) (En parte, hay que dejar oraciones únicas).
  
- Sobre parseo de oraciones y balanceo de clases
- sobre multiclases, que nos conviene?

# TODO
- chequear balanceo de clases
- realizar dt y svm o knn
- falta atributo de signo de pregunta en el texto